# 01 - Data Ingestion

**Objective:** Load the Adult Income dataset (train + test), handle missing values ("?"), encode target, one-hot encode categoricals, align train/test features, and save.

**Dataset:** UCI Adult Income — 32,561 train / 16,281 test, 14 features, binary target (>50K / <=50K)

**Steps:**
1. Load raw train and test CSV files
2. Handle missing values and strip whitespace from strings
3. Encode target as binary
4. One-hot encode categorical variables
5. Align train and test columns
6. Save processed data


In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path

print("Libraries imported successfully")


### Dataset Attributes

Understanding each column helps you make better decisions during cleaning and feature engineering.

- **age:** Age of the individual (numeric)
- **workclass:** Employment type — categorical with leading whitespace (e.g., " State-gov")
- **fnlwgt:** Final sampling weight (numeric, used by Census Bureau for population estimates)
- **education:** Highest education level attained (categorical, e.g., "Bachelors", "HS-grad")
- **education-num:** Education level as a number (numeric, 1-16)
- **marital-status:** Marital status (categorical)
- **occupation:** Job type (categorical)
- **relationship:** Family role (categorical, e.g., "Husband", "Not-in-family")
- **race:** Race (categorical)
- **sex:** Sex (categorical, with leading whitespace: " Male", " Female")
- **capital-gain:** Capital gains (numeric, many zeros)
- **capital-loss:** Capital losses (numeric, many zeros)
- **hours-per-week:** Hours worked per week (numeric)
- **native-country:** Country of origin (categorical)
- **income:** Target — ">50K" or "<=50K" in train, ">50K." or "<=50K." in test (note trailing dot)


In [ ]:
# TODO: Load the raw CSV files into DataFrames
# The Adult dataset comes as two files: adult.data (train) and adult.test (test).
# Both have no header row, so we assign column names ourselves.
# The test file has an extra "|" separator line at the top, so use skiprows=1.

RAW_DIR = Path("../data/raw")

column_names = [
    "age", "workclass", "fnlwgt", "education", "education-num",
    "marital-status", "occupation", "relationship", "race", "sex",
    "capital-gain", "capital-loss", "hours-per-week", "native-country", "income"
]

# TODO: Load train and test
# Use pd.read_csv() with header=None and names=column_names.
# For the test file, add skiprows=1 to skip the separator line.
# After loading, check the shapes to confirm they match expectations.

# train = pd.read_csv(RAW_DIR / "adult.data", header=None, names=column_names)
# test = pd.read_csv(RAW_DIR / "adult.test", header=None, names=column_names, skiprows=1)
# print(f"Train: {train.shape}, Test: {test.shape}")
# print(train.head())
# test.info()


In [ ]:
# TODO: Handle missing values and strip whitespace
# String columns in this dataset have leading whitespace (e.g., " State-gov", " Male").
# These spaces cause pd.get_dummies() to treat " State-gov" and "State-gov" as different values.
# Missing values are encoded as "?" — strip first, then replace "?" with pd.NA and drop.

# There are many categorical columns, so loop over all object-type columns.
# df[col].str.strip() removes leading/trailing spaces.
# Then df.replace("?", pd.NA, inplace=True) marks missing entries.
# Finally df.dropna() removes any row with at least one missing value.

# for col in train.columns:
#     if train[col].dtype == "object":
#         train[col] = train[col].str.strip()
#         test[col] = test[col].str.strip()

# train.replace("?", pd.NA, inplace=True)
# test.replace("?", pd.NA, inplace=True)

# train = train.dropna().reset_index(drop=True)
# test = test.dropna().reset_index(drop=True)
# print(f"Train after cleaning: {train.shape}, Test after cleaning: {test.shape}")


In [ ]:
# TODO: Encode target variable as binary
# The train file uses ">50K" and "<=50K"; the test file uses ">50K." and "<=50K." (trailing dot).
# Use a lambda that checks if ">50K" appears anywhere in the string — this handles both cases.

# train["income"] = train["income"].apply(lambda x: 1 if ">50K" in str(x) else 0)
# test["income"] = test["income"].apply(lambda x: 1 if ">50K" in str(x) else 0)

# TODO: Check class distribution for both sets
# The income dataset is imbalanced: roughly 75% <=50K, 25% >50K.
# This matters because accuracy can be misleading with imbalanced classes.

# print("Train distribution:")
# print(train["income"].value_counts(normalize=True))
# print("Test distribution:")
# print(test["income"].value_counts(normalize=True))


In [ ]:
# TODO: One-hot encode categorical variables
# Many features are categorical (workclass, education, marital-status, occupation,
# relationship, race, sex, native-country). Models need numeric inputs, so we
# convert each category into its own binary column with pd.get_dummies().
# Set drop_first=True to avoid the dummy variable trap (perfect multicollinearity).

# train_enc = pd.get_dummies(train, drop_first=True)
# test_enc = pd.get_dummies(test, drop_first=True)

# TODO: Align train and test columns
# The test set might have fewer category values than the train set (or miss some entirely).
# Use .align() to ensure both DataFrames have the same columns, filling missing ones with 0.

# train_enc, test_enc = train_enc.align(test_enc, join="left", axis=1, fill_value=0)
# print(f"Train encoded: {train_enc.shape}, Test encoded: {test_enc.shape}")
# print(test_enc.columns.tolist())


In [ ]:
# TODO: Save processed data
# Save both train and test to the processed directory.
# Use index=False so we don't add a row-number column.

PROCESSED_DIR = Path("../data/processed")
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# train_enc.to_csv(PROCESSED_DIR / "clean_train.csv", index=False)
# test_enc.to_csv(PROCESSED_DIR / "clean_test.csv", index=False)
# print("Processed data saved successfully")


In [ ]:
# TODO: Verify the saved files by loading them back
# This confirms the files weren't corrupted during the write process.
# Compare the shapes of the loaded data to the originals.

# train_check = pd.read_csv(PROCESSED_DIR / "clean_train.csv")
# test_check = pd.read_csv(PROCESSED_DIR / "clean_test.csv")
# assert train_check.shape == train_enc.shape, "Train shape mismatch"
# assert test_check.shape == test_enc.shape, "Test shape mismatch"
# print("Verification passed: both files round-trip correctly")


## 2. Relational Data — Joining Tables

In real-world projects, data is rarely handed to you in a single tidy CSV. 
It is often spread across multiple tables in a database — a **fact** table 
and several **dimension** tables, linked by key columns.

To build your feature matrix you need to **join** these tables back together.

In this section you will work with a relational version of the Adult Census Income data,
split into three CSV files that simulate a government database:

| File | Content | Key column |
|------|---------|------------|
| `personal.csv` | Demographics (age, sex, race, native-country) | `sample_id` |
| `employment.csv` | Work/education records | `employment_id` |
| `census.csv` | Census financial data + income target | `census_id` |

Some IDs appear in only some tables — exactly like a real database where 
employment records or tax data may be missing for certain individuals. 
Your job is to re-assemble the full dataset.

In [ ]:
# Load the three relational tables
RELATIONAL_DIR = Path("../data/raw/relational")

personal = pd.read_csv(RELATIONAL_DIR / "personal.csv")
employment = pd.read_csv(RELATIONAL_DIR / "employment.csv")
census = pd.read_csv(RELATIONAL_DIR / "census.csv")

# TODO: Inspect each table
# Check the shape and first few rows.
# How many IDs does each table have? Are the ID ranges the same?

# print(f"Personal: {personal.shape}")
# print(f"Employment: {employment.shape}")
# print(f"Census: {census.shape}")
# print(personal.head())
# print(employment.columns.tolist())


In [ ]:
# TODO: Merge personal with employment records
# Use pd.merge() with left_on="sample_id" and right_on="employment_id".
# Use a LEFT JOIN to keep all individuals.
# Count how many rows have NaN for employment columns.

# merged = pd.merge(personal, employment, left_on="sample_id", right_on="employment_id", how="left")
# print(f"After merge: {merged.shape}")
# nan_emp = merged['workclass'].isna().sum()
# print(f"Missing employment: {nan_emp}")

# TODO: Now try an INNER JOIN. How many rows do you lose? Why?


In [ ]:
# TODO: Merge in the census financial table
# Chain a second merge to bring in census data + income target.

# full = pd.merge(merged, census, left_on="sample_id", right_on="census_id", how="left")
# print(full.head())

# TODO: Check how many rows are missing census data
# missing_cen = full['fnlwgt'].isna().sum()
# print(f"Missing census: {missing_cen}")


In [ ]:
# TODO: Compare to the original single-table shape
# The original adult.data has 32561 rows with all 14 features + income target.
# After splitting and rejoining, how many complete rows do you have?

# complete = full.dropna()
# print(f"Complete rows after join: {len(complete)}")

# HINT: This number is smaller than 32561. Why?
# Can you determine which table has the most missing records?


### Exercises

1. **String edge cases**: The encoding logic uses `">50K" in str(x)`. What if a test row already says `">50K"` without the trailing dot? Would the encoding still produce the correct result? What about a hypothetical value like `">50K."` (with dot) that contains the substring? (HINT: Test with a small DataFrame.)
2. **Which JOIN?**: You used LEFT JOINs above. Replace them with INNER JOINs. How many rows do you lose overall? Why?
3. **Fake data detective**: The employment and census tables contain artificially generated rows (IDs > 32561). How many can you find? What heuristic did you use?
4. **Missing value experiment**: Intentionally set a few `workclass` values to NaN before the `.dropna()`. Compare `dropna()` vs `fillna(train["workclass"].mode()[0])`. How does each strategy affect the final shape and the class distribution?
5. **Feature engineering**: Create a new column `capital_total = capital-gain - capital-loss`. Use `groupby("income")["capital_total"].describe()` to see if net capital correlates with income >50K. Does it?
6. **What would break?**: If the raw `adult.data` file had an extra column appended to it (e.g., "education-level" as a duplicate of education-num), which cell in this notebook would fail first? What error would it produce?
7. **Merge on native-country**: Try merging `personal` with `census` on `native-country` instead of sample_id. What happens to the row count? Why is merging on non-key columns dangerous?
8. **Extend to test data**: The test file `adult.test` has the same structure (with the trailing-dot quirk). Could you repeat this exercise with the test set using the same relational tables? What would need to change?